In [110]:
import pandas as pd

df = pd.read_csv('mismatch_summary.tsv', sep='\t')

scan	DB_search_score	precursor_score	DB_proteins	precursor_proteins	precursor_better	charge	peptide	peptide_demod	peptide_length	reason_mismatch	MinNTermAdd	minNTermSubtract	MinCTermAdd	minCTermSubtract
6352	-25.036301	106.825996	Q86Y38	P61224;E7ESV4;F5GWU8;F5GX62;F5GYB5;F5H004;F5H077;F5H0B7;F5H491;F5H500;F5H6R7;F5H7Y6	True	2	KQVEVDAQQC+57.021MLEILDTAGTEQ	KQVEVDAQQCMLEILDTAGTEQ	22	non_standard_modification	0	0	5	21
6353	-23.2418	103.260002	Q9H0U3-2;Q9H0U3;A0A087WU53;A0A8I5KUC4;A0A8I5KY62;A0A8I5KYH1;A0A8I5QJJ8;A0A8I5QJM4;A0A8I5QKX7	P61224;E7ESV4;F5GWU8;F5GX62;F5GYB5;F5H004;F5H077;F5H0B7;F5H491;F5H500;F5H6R7;F5H7Y6	True	2	KQVEVDAQQC+57.021MLEILDTAGTEQ	KQVEVDAQQCMLEILDTAGTEQ	22	non_standard_modification	0	0	5	21

In [111]:
# df_filtered = df[(df['precursor_better'] == True) & (df['precursor_proteins'].str.contains(';') == False)]

In [112]:
# df_filtered.head()

In [113]:
prec_protein = set(df['precursor_proteins'].dropna().str.split(r'[;-]').str[0])
# print(f"Number of unique precursor proteins: {len(prec_protein)}")
print(f"Number of unique precursor proteins: {len(prec_protein)}")

Number of unique precursor proteins: 331


In [114]:
import re

# Create a dictionary to store the data for each precursor protein
#iterate through filtered df each row, make a new df with columns: precursor_protein, precursor_id_number - this is the number of rows(peptides) that has that protein in precursor_proteins, [(evidence_peptides,scan_number) - this is a tuple with multiple peptides and scan ]

protein_data = {}

for idx, row in df.iterrows():
    # Create a dictionary to store the data for each precursor protein
    #iterate through filtered df each row, make a new df with columns: precursor_protein, precursor_id_number - this is the number of rows(peptides) that has that protein in precursor_proteins, [(evidence_peptides,scan_number) - this is a tuple with multiple peptides and scan ]

    proteins = row['precursor_proteins']
    if pd.isna(proteins):
        continue
    protein = re.split(r'[;-]', proteins)[0]
    peptide = row['peptide_demod']
    if len(peptide) <9:
        continue
    scan = row['scan']
    reason_mismatch = row['reason_mismatch']
    evidence = False

    
    if protein not in protein_data:
        protein_data[protein] = {
            'precursor_protein': protein,
            'precursor_id_number': 0,
            'evidence_peptides_scans': []
        }
    
    protein_data[protein]['precursor_id_number'] += 1
    protein_data[protein]['evidence_peptides_scans'].append((peptide, scan,reason_mismatch,evidence))

for protein, data in protein_data.items():
    # print(data)
    peptides_temp = []
    for peptide, _, _, _ in protein_data[protein]['evidence_peptides_scans']:
        peptides_temp.append(peptide)
    # print(peptides_temp)
    for p1 in peptides_temp:
        if len(p1) < 9:
            peptides_temp.remove(p1)
            continue
        for p2 in peptides_temp:
            if p1 in p2 and p1 != p2:
                peptides_temp.remove(p2)
                continue
    
    peptides_temp = list(set(peptides_temp))
    
    # print(peptides_temp)

    data['two_noncontained_le9'] = len(peptides_temp)

    for peptide, _, _, evidence in protein_data[protein]['evidence_peptides_scans']:
        if peptide in peptides_temp:
            for i, (pep, scan, reason, _) in enumerate(protein_data[protein]['evidence_peptides_scans']):
                if pep == peptide:
                    protein_data[protein]['evidence_peptides_scans'][i] = (pep, scan, reason, True)
# Create the new dataframe
df_protein_summary = pd.DataFrame(protein_data.values())

In [115]:
df_protein_summary.to_csv('precursor_protein_level.tsv', sep='\t', index=False)